# Nonlinear Regression

In the `Wage` data, a straight line forces the effect of age to be the same at
20, 40, and 60. The scatter suggests a more flexible question: how can we let
the conditional mean rise, flatten, or fall without chasing every random
fluctuation?

Squared loss still targets the conditional mean; what changes is the class of
functions used to approximate it. We will build two routes to flexibility from
scratch---one global and one local---and then choose how flexible each fit should
be using held-out data:

1. polynomial regression, the parametric route;
2. the local average, and why it is rough;
3. $k$-nearest neighbours;
4. kernel regression, the smooth-weight version;
5. local linear regression as weighted least squares;
6. the curse of dimensionality;
7. choosing the complexity knob by cross-validation.

In [ ]:
# Colab opens this notebook on its own, so fetch the data file that sits
# beside it in the course repository. Running locally, this does nothing.
from pathlib import Path as _Path
import urllib.request as _req
_RAW = 'https://raw.githubusercontent.com/ZhuoranYang/sds265-fall26/main/demos/06-nonlinear-regression/'
if not _Path('Wage.csv').exists():
    _req.urlretrieve(_RAW + 'Wage.csv', 'Wage.csv')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 26506
rng = np.random.default_rng(SEED)
wage = pd.read_csv('Wage.csv')
age = wage['age'].to_numpy(float)
response = wage['wage'].to_numpy(float)
grid = np.linspace(age.min(), age.max(), 300)
print('Wage rows:', len(wage), '| age range:', age.min(), '-', age.max())

## 1. A global route: polynomial regression

A line may miss curvature in the age--wage relationship. One way to add it is
to give ordinary least squares the transformed features
$1,z,z^2,\ldots,z^d$. The resulting polynomial is nonlinear in age but linear
in its fitted coefficients. Centering and scaling age improves numerical
conditioning without changing the function class.

The next cell fits degrees 1, 4, and 10 to the same observations. Compare how
quickly flexibility grows with degree and pay particular attention to behavior
near the edges, where high-degree polynomials have the least support. The call
`np.linalg.lstsq(train_design, y_train, rcond=None)[0]` solves ordinary least
squares on the transformed design; `[0]` selects the coefficient vector from
NumPy's four returned objects.

In [ ]:
def polynomial_design(x, degree, center, scale):
    z = (x-center)/scale
    return np.column_stack([z**j for j in range(degree+1)])

def polynomial_fit_predict(x_train, y_train, query, degree):
    center, scale = x_train.mean(), x_train.std()
    train_design = polynomial_design(x_train, degree, center, scale)
    # lstsq accepts this rectangular design; [0] is the coefficient vector.
    beta = np.linalg.lstsq(train_design, y_train, rcond=None)[0]
    return polynomial_design(query, degree, center, scale) @ beta

plt.figure(figsize=(5.5, 3.2))
plt.scatter(age, response, s=5, alpha=.12)
for degree in (1, 4, 10):
    plt.plot(grid, polynomial_fit_predict(age, response, grid, degree),
             label=f'degree {degree}')
plt.xlabel('age'); plt.ylabel('wage'); plt.legend();

## 2. A local route: average nearby responses

A polynomial makes one global shape assumption. A local method instead assumes
only that people with nearby ages have similar conditional means. To estimate
$m(x)$, average the responses for rows with $|x_i-x|\le h$.

The left panel of the next cell makes one prediction at age 40 using $h=5$ and
highlights exactly which workers contribute. The right panel repeats the same
calculation at every query using $h=0.75$. Compare the stability of the single
wide-window mean with the roughness of the narrow-window curve.

In [ ]:
def local_average(x_train, y_train, query, h):
    out = np.full(len(query), np.nan)
    for j, q in enumerate(query):
        inside = np.abs(x_train-q) <= h
        if inside.any():
            out[j] = y_train[inside].mean()
    return out

inside_40 = np.abs(age-40) <= 5
print(f'at age 40 with h=5: {inside_40.sum()} of {len(age)} rows, estimate {response[inside_40].mean():.1f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
axes[0].scatter(age[~inside_40], response[~inside_40], s=5, alpha=.10, color='0.6')
axes[0].scatter(age[inside_40], response[inside_40], s=6, alpha=.35)
axes[0].axvspan(35, 45, alpha=.08)
axes[0].hlines(response[inside_40].mean(), 35, 45, lw=2, color='C3')
axes[0].set(title=f'one query: mean of {inside_40.sum()} points', xlabel='age', ylabel='wage')
axes[1].scatter(age, response, s=5, alpha=.10, color='0.6')
axes[1].plot(grid, local_average(age, response, grid, .75), lw=1, color='C3')
axes[1].set(title='every query, h=0.75: rough', xlabel='age')
plt.tight_layout();

## 3. Adapt the window with $k$-nearest neighbours

A fixed radius can contain many observations at common ages and very few near
the boundary. $k$-nearest neighbours fixes the *count* instead, so the window
automatically widens where observations are scarce.

The next cell fits $k=5$, 50, and 500. Compare how the neighborhood size trades
local detail for averaging: small $k$ is flexible and noisy, whereas large $k$
uses less local information and approaches the overall mean.

In [ ]:
def knn_regression(x_train, y_train, query, k):
    order = np.argsort(np.abs(query[:, None]-x_train[None, :]), axis=1)
    return y_train[order[:, :k]].mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.0), sharey=True)
for ax, k in zip(axes, (5, 50, 500)):
    ax.scatter(age, response, s=5, alpha=.10, color='0.6')
    ax.plot(grid, knn_regression(age, response, grid, k), lw=1.3)
    ax.set(title=f'k={k}', xlabel='age')
axes[0].set_ylabel('wage'); plt.tight_layout();

The neighbor count is the complexity knob in
reverse: small $k$ gives a flexible, noisy curve; large $k$ averages more of
the sample and eventually approaches a constant.

## 4. Smooth the cutoff with kernel regression

The previous local rules change abruptly when a row enters or leaves the
neighborhood: a point just inside counts fully, while one just outside does not
count. Kernel regression replaces that hard cutoff by a weight that decays with
distance.

The next cell uses Gaussian bandwidths $h=1.5$, 5, and 15; compare their
roughness and bias. It also replaces the Gaussian by a box kernel to verify that
the original local average is a special case of the same weighted-average
formula.

In [ ]:
def kernel_regression(x_train, y_train, query, h):
    weights = np.exp(-.5*((query[:, None]-x_train[None, :])/h)**2)
    return (weights@y_train)/weights.sum(1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.0), sharey=True)
for ax, h in zip(axes, (1.5, 5.0, 15.0)):
    ax.scatter(age, response, s=5, alpha=.10, color='0.6')
    ax.plot(grid, kernel_regression(age, response, grid, h), lw=1.3)
    ax.set(title=f'h={h}', xlabel='age')
axes[0].set_ylabel('wage'); plt.tight_layout();

# The box kernel recovers the local average exactly.
box = np.abs(grid[:, None]-age[None, :]) <= 5
print('box kernel equals local average:',
      np.allclose((box@response)/box.sum(1), local_average(age, response, grid, 5.)))

The bandwidth $h$ plays the same role as $k$.
Small $h$ follows local fluctuations; large $h$ lowers variance by averaging
observations that are farther from the query.

### Make bias and variance visible

Repeatedly drawing new datasets from a known curve lets us measure two
quantities that a single `Wage` dataset cannot reveal: bias, the systematic
difference between the average fitted curve and the truth, and variance, how
much the fitted curve changes from sample to sample.

The next cell adapts that experiment into one compact plot. For each bandwidth,
it draws 200 training samples, refits the Gaussian kernel smoother, and averages
the squared bias and variance over the query grid. Look for the opposite
directions of the two curves and the intermediate bandwidth favored by their
sum.

In [ ]:
def smoothing_decomposition(bandwidths, trials=200, sample_size=50,
                                noise_sd=.25, seed=265):
    simulation_rng = np.random.default_rng(seed)
    query = np.linspace(-.9, .9, 80)
    truth = 3 * query**3
    rows = []
    for h in bandwidths:
        estimates = []
        for _ in range(trials):
            x_sample = np.sort(simulation_rng.uniform(-1, 1, sample_size))
            y_sample = 3*x_sample**3 + noise_sd*simulation_rng.standard_normal(sample_size)
            estimates.append(kernel_regression(x_sample, y_sample, query, h))
        estimates = np.asarray(estimates)
        squared_bias = np.mean((estimates.mean(axis=0) - truth)**2)
        variance = np.mean(estimates.var(axis=0, ddof=1))
        rows.append((h, squared_bias, variance,
                     squared_bias + variance + noise_sd**2))
    return pd.DataFrame(rows, columns=["bandwidth", "squared_bias", "variance", "risk"])

bias_variance = smoothing_decomposition([.03, .06, .10, .20, .35])
ax = bias_variance.plot(x="bandwidth", y=["squared_bias", "variance", "risk"],
                        marker="o", figsize=(5.5, 3.2))
ax.set(ylabel="average squared error component",
       title="repeated samples reveal the bias--variance trade-off")
bias_variance.round(4)

A narrow bandwidth keeps the average curve close
to the truth but makes the estimate sensitive to the particular sample. A wide
bandwidth produces repeatable curves but smooths away real structure. The total
risk is smallest between those extremes; cross-validation will estimate that
choice when the true regression function is unknown.

## 5. Correct boundary behavior with a local line

A weighted average fits a local constant. Near the youngest and oldest ages,
almost all neighbors lie on one side of the query, so that constant is pulled
toward the interior. Fitting a local *line* lets the one-sided data determine a
slope as well as a level:
$(Z^\top W Z)\widehat c=Z^\top W y$ with
$\widehat m(x_0)=\widehat a$. For every query, the code passes this square
$2\times2$ normal-equation system to `np.linalg.solve`; `[0]` then selects the
intercept from the returned two-coefficient vector. We use `solve` instead of
forming $(Z^\top W Z)^{-1}$ explicitly.

The next cell compares local constant and local linear fits using the same
bandwidth $h=5$, then reports their average difference at the left boundary.
Keeping the weights fixed isolates what changes when we fit a line instead of a
constant.

In [ ]:
def local_linear(x_train, y_train, query, h):
    out = np.empty(len(query))
    for j, q in enumerate(query):
        w = np.exp(-.5*((x_train-q)/h)**2)
        Z = np.column_stack([np.ones_like(x_train), x_train-q])
        ZtW = Z.T*w
        system_matrix = ZtW @ Z
        right_hand_side = ZtW @ y_train
        local_coefficients = np.linalg.solve(system_matrix, right_hand_side)
        out[j] = local_coefficients[0]  # intercept at x_train-q=0 is the prediction
    return out

plt.figure(figsize=(5.5, 3.2))
plt.scatter(age, response, s=5, alpha=.10, color='0.6')
plt.plot(grid, kernel_regression(age, response, grid, 5.), label='local constant')
plt.plot(grid, local_linear(age, response, grid, 5.), label='local linear')
plt.xlabel('age'); plt.ylabel('wage'); plt.legend()
edge = grid < 22
print('mean gap at the left boundary:',
      round(float(np.mean(local_linear(age,response,grid,5.)[edge]
                          - kernel_regression(age,response,grid,5.)[edge])), 2))

The sizable boundary gap shows that the local
model---constant versus line---matters even when the bandwidth and observations
are unchanged. It does not by itself tell us which curve predicts better; that
requires validation on observations not used to fit the curve.

## 6. The curse of dimensionality

Local regression works only when a query has genuinely nearby training points.
Adding irrelevant features spreads distances across more coordinates, so the
five closest points can be far away in the coordinate that matters.

The next simulation makes this failure measurable: only the first feature
affects the response, and every added feature is pure noise. We compare 5-NN,
which uses all coordinates to define distance, with linear regression, whose
global form does not require nearby observations. Track how test error changes
as $p$ grows even though the true regression function never changes.

In [ ]:
# seed 265 matches the slide figure
def curse_experiment(dims=(1,2,3,4,5,10,20), n_train=50, n_test=500, reps=60, k=5, seed=265):
    gen = np.random.default_rng(seed)
    f = lambda Z: np.sin(2.0*Z[:,0]) + .3*Z[:,0]
    rows = []
    for p in dims:
        lin_r, knn_r = [], []
        for _ in range(reps):
            Xtr, Xte = gen.uniform(-1,1,(n_train,p)), gen.uniform(-1,1,(n_test,p))
            ytr, yte = f(Xtr) + .2*gen.standard_normal(n_train), f(Xte)
            A = np.column_stack([np.ones(n_train), Xtr])
            # Rectangular least-squares problem; [0] selects the coefficients.
            beta = np.linalg.lstsq(A, ytr, rcond=None)[0]
            lin_r.append(np.mean((yte - np.column_stack([np.ones(n_test), Xte])@beta)**2))
            d = np.linalg.norm(Xte[:,None,:]-Xtr[None,:,:], axis=2)
            knn_r.append(np.mean((yte - ytr[np.argsort(d,axis=1)[:,:k]].mean(1))**2))
        rows.append((p, np.mean(knn_r), np.mean(lin_r)))
    return pd.DataFrame(rows, columns=['p','knn_MSE','linear_MSE'])

curse = curse_experiment()
plt.figure(figsize=(5.5, 3.0))
plt.loglog(curse.p, curse.knn_MSE, 'o-', label='k-nearest neighbours (k=5)')
plt.loglog(curse.p, curse.linear_MSE, 'o-', label='linear regression')
plt.xlabel('number of features p (only the first matters)'); plt.ylabel('test MSE'); plt.legend()
print(curse.round(3).to_string(index=False))
print(f"\nKNN error grows {curse.knn_MSE.iloc[-1]/curse.knn_MSE.iloc[0]:.0f}x from p=1 to p=20, "
      f"linear {curse.linear_MSE.iloc[-1]/curse.linear_MSE.iloc[0]:.1f}x")

The irrelevant coordinates damage 5-NN much
more quickly because they corrupt the definition of a neighbor. This is the
curse of dimensionality: a local method needs dramatically more data to maintain
the same neighborhood quality as dimension grows. Feature selection or dimension
reduction can therefore matter before fitting a local smoother.

## 7. Choose flexibility by cross-validation

Degree, $k$, and $h$ look different, but all three control the bias--variance
trade-off. We choose them the same way $\lambda$ was chosen in Lecture 4.

The next cells hide 25% of the workers as a final test set, use the *same five
training folds* to score every candidate procedure, and mark the smallest mean
validation error. Only after choosing each knob do we refit on all training rows
and use the test responses once. Compare predictive error across methods; do not
interpret different numerical knob values as comparable amounts of flexibility.

In [ ]:
order = rng.permutation(len(age))
cut = int(.75*len(order))
train_idx, test_idx = order[:cut], order[cut:]
x_tr, y_tr = age[train_idx], response[train_idx]
x_te, y_te = age[test_idx], response[test_idx]
folds = np.array_split(rng.permutation(len(x_tr)), 5)

def cross_validate(fit_predict, values):
    rows = []
    for v in values:
        scores = []
        for f in range(len(folds)):
            val = folds[f]
            tr = np.concatenate([folds[g] for g in range(len(folds)) if g != f])
            pred = fit_predict(x_tr[tr], y_tr[tr], x_tr[val], v)
            scores.append(np.mean((y_tr[val]-pred)**2))
        rows.append((v, np.mean(scores)))
    return pd.DataFrame(rows, columns=['value','cv_MSE'])

poly_cv = cross_validate(polynomial_fit_predict,
                         range(1, 11))
knn_cv  = cross_validate(lambda xa, ya, xb, k: knn_regression(xa, ya, xb, k),
                         [2, 5, 10, 25, 50, 100, 200, 400])
kern_cv = cross_validate(lambda xa, ya, xb, h: kernel_regression(xa, ya, xb, h),
                         [1., 2., 3., 5., 8., 12., 20., 40.])

for name, table in [('degree', poly_cv), ('k', knn_cv), ('h', kern_cv)]:
    best = table.loc[table.cv_MSE.idxmin()]
    print(f'{name:7s} chosen = {best.value:<6g}  CV MSE = {best.cv_MSE:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.0))
for ax, table, label in zip(
    axes, [poly_cv, knn_cv, kern_cv], ['polynomial degree', 'neighbor count k', 'bandwidth h']
):
    ax.plot(table.value, table.cv_MSE, 'o-')
    best = table.loc[table.cv_MSE.idxmin()]
    ax.scatter(best.value, best.cv_MSE, marker='*', s=110, color='C3', zorder=3)
    ax.set(xlabel=label, ylabel='five-fold CV MSE')
plt.tight_layout();

In [ ]:
# Refit each at its chosen value on all training rows, then score the test set once.
best_degree = int(poly_cv.loc[poly_cv.cv_MSE.idxmin(),'value'])
best_k      = int(knn_cv.loc[knn_cv.cv_MSE.idxmin(),'value'])
best_h      = float(kern_cv.loc[kern_cv.cv_MSE.idxmin(),'value'])

final = {
    f'polynomial (degree {best_degree})': polynomial_fit_predict(x_tr, y_tr, x_te, best_degree),
    f'k-nearest neighbours (k={best_k})': knn_regression(x_tr, y_tr, x_te, best_k),
    f'kernel regression (h={best_h:g})':  kernel_regression(x_tr, y_tr, x_te, best_h),
    'constant baseline':                  np.full(len(x_te), y_tr.mean()),
}
pd.DataFrame({'test MSE': {k: np.mean((y_te-v)**2) for k, v in final.items()}}).round(1)

The final table answers a prediction question on
this particular split; it does not declare one family universally superior.
Cross-validation has adapted the flexibility of each family before the test
comparison, and the constant baseline shows how much is gained by using age at
all.

## Verify the from-scratch implementations

Before interpreting the comparisons, we check that our custom algorithms match
standard library implementations. The next cell compares degree-4 polynomial
predictions directly and then compares KNN predictions, where tied ages require
one additional caveat.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsRegressor

library = make_pipeline(PolynomialFeatures(4), LinearRegression()).fit(x_tr[:, None], y_tr)
scratch = polynomial_fit_predict(x_tr, y_tr, x_te, 4)
print('polynomial, max difference:', np.max(np.abs(library.predict(x_te[:, None])-scratch)))
assert np.allclose(library.predict(x_te[:, None]), scratch, atol=1e-6)

lib_knn = KNeighborsRegressor(n_neighbors=best_k).fit(x_tr[:, None], y_tr)
knn_difference = (lib_knn.predict(x_te[:, None])
                  - knn_regression(x_tr, y_tr, x_te, best_k))
print('knn, mean absolute difference:', np.mean(np.abs(knn_difference)))
print('knn, maximum difference:', np.max(np.abs(knn_difference)))

The polynomial implementations agree up to
floating-point error. KNN need not match row for row because age is recorded in
whole years: when the $k$th distance is tied, our stable sort and scikit-learn's
neighbor search may choose different workers at that same distance. This is a
tie-breaking difference, not a different definition of distance.

## Try It

For each change, predict the fitted curve or error before running the cell.

1. Set polynomial degree to 20. Compare the curve inside the age range with its extrapolation to ages 10 and 95.
2. Set $k=1$ and then $k=n$. Which training observations are interpolated, and what constant appears at the other extreme?
3. Replace the Gaussian kernel by the box kernel. Which fitted curve becomes exactly the local average?
4. In the curse-of-dimensionality simulation, increase the training size from 50 to 500. Does that remove the curse or only postpone it?